# EarthScape ML Exploration
Structured notebook for data-quality filtering, anomaly detection training, and temperature trend modeling.

## 1) Imports

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

## 2) Setup Paths & Load Raw Dataset

In [ ]:
repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
raw_dir = repo_root / 'data' / 'raw'
model_dir = repo_root / 'backend' / 'ml' / 'models'
model_dir.mkdir(parents=True, exist_ok=True)

csv_files = sorted(raw_dir.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError('No CSV files in data/raw. Run 01_dataset_download.ipynb first.')

print(f'Reading dataset: {csv_files[0].name}')
df = pd.read_csv(csv_files[0])
df.head()

## 3) Date Features

In [ ]:
df['dt'] = pd.to_datetime(df['dt'])
df['year'] = df['dt'].dt.year
df['month'] = df['dt'].dt.month
df[['dt', 'year', 'month']].head()

## 4) Dynamic Data Quality Analysis

In [ ]:
print('\n--- Running Dynamic Data Quality Analysis ---')

if 'AverageTemperatureUncertainty' in df.columns:
    yearly_stats = df.groupby('year').agg(
        avg_uncertainty=('AverageTemperatureUncertainty', 'mean'),
        missing_pct=('AverageTemperature', lambda x: x.isna().mean() * 100)
    ).reset_index()

    quality_condition = (yearly_stats['avg_uncertainty'] < 0.75) & (yearly_stats['missing_pct'] < 2.0)
    valid_years = yearly_stats[quality_condition]

    if not valid_years.empty:
        best_start_year = int(valid_years['year'].min())
        print(f'Target located! Average uncertainty drops below 0.75°C starting in: {best_start_year}')
    else:
        best_start_year = 1900
        print('Uncertainty threshold not met cleanly. Falling back to default: 1900')
else:
    best_start_year = 1900
    print('Uncertainty column missing. Falling back to default: 1900')

yearly_stats.head() if 'yearly_stats' in locals() else best_start_year

## 5) Apply Filter + Clean Columns

In [ ]:
df = df[df['year'] >= best_start_year].copy()
print(f'Filtered dataset to include rows from {best_start_year} to present. Current row count: {len(df):,}')

df = df.rename(columns={'AverageTemperature': 'temperature_c', 'Latitude': 'latitude', 'Longitude': 'longitude'})

def parse_latlon(value):
    text = str(value).strip().upper()
    num = ''.join(ch for ch in text if ch.isdigit() or ch in '.-+')
    if not num:
        return 0.0
    val = float(num)
    if text.endswith('S') or text.endswith('W'):
        return -val
    return val

df['temperature_c'] = pd.to_numeric(df['temperature_c'], errors='coerce')
df['latitude_num'] = df['latitude'].map(parse_latlon)
df['longitude_num'] = df['longitude'].map(parse_latlon)

train_clean = df[['year', 'month', 'temperature_c', 'latitude_num', 'longitude_num']].dropna()
print(f'Final clean training dataset size: {len(train_clean):,}')
train_clean.head()

## 6) Train Isolation Forest (Anomaly Detection)

In [ ]:
print('\nTraining Isolation Forest...')
iso_model = IsolationForest(random_state=42, contamination=0.02)
iso_model.fit(train_clean[['temperature_c', 'latitude_num', 'longitude_num']].values)

isolation_model_path = model_dir / 'isolation_forest.pkl'
joblib.dump(iso_model, isolation_model_path)
print(f'Saved anomaly model to {isolation_model_path}')

## 7) Train Spark Linear Regression (Trend Prediction)

In [ ]:
import os
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# 1. CYCLICAL FEATURE ENGINEERING: Map month to geometric sine/cosine waves
print("Engineering cyclical seasonal features in pandas...")
train_clean['sin_month'] = np.sin(2 * np.pi * train_clean['month'] / 12.0)
train_clean['cos_month'] = np.cos(2 * np.pi * train_clean['month'] / 12.0)

# Buffer the engineered data directly to disk to preserve memory
print("Buffering updated data to disk to preserve cloud RAM...")
train_clean.to_csv('spark_staging_buffer.csv', index=False)

print('\nStarting memory-allocated PySpark Session...')
spark = SparkSession.builder \
    .master('local[*]') \
    .appName('earthscape-notebook-trend') \
    .config("spark.driver.memory", "6g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

# Keep system logs totally silent
spark.sparkContext.setLogLevel("WARN")

print('Streaming data from disk buffer into Spark environment...')
sdf = spark.read.csv('spark_staging_buffer.csv', header=True, inferSchema=True)
sdf = sdf.withColumn('time_idx', F.col('year') + (F.col('month') - 1) / 12.0)

# 2. UPGRADED FEATURE ASSEMBLER: Inject sin_month and cos_month into features array
print('Assembling features (including cyclical seasonal features)...')
feature_cols = ['time_idx', 'latitude_num', 'longitude_num', 'sin_month', 'cos_month']
assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')
train_df = assembler.transform(sdf).select('features', 'temperature_c')

# 3. Apply the 80/20 Train/Test Split
train_data, test_data = train_df.randomSplit([0.8, 0.2], seed=42)
print(f"Data Split Complete -> Training Rows: {train_data.count():,} | Testing Rows: {test_data.count():,}")

print('Fitting Upgraded Linear Regression model on training split (80%)...')
lr = LinearRegression(featuresCol='features', labelCol='temperature_c')
lr_model = lr.fit(train_data)

# 4. Run predictions on the unseen test data (20%)
print('Evaluating upgraded model performance metrics on test split...')
predictions = lr_model.transform(test_data)

# 5. Calculate accuracy metrics
evaluator_rmse = RegressionEvaluator(labelCol="temperature_c", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="temperature_c", predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print("\n=============================================")
print("   UPGRADED LINEAR REGRESSION METRICS        ")
print("=============================================")
print(f" Root Mean Squared Error (RMSE): {rmse:.4f} °C")
print(f" R-squared (R²) Score: {r2:.4f}")
print("=============================================\n")

# 6. Save the model folder (overwriting old folder safely)
spark_model_path = model_dir / 'spark_linear_regression'
lr_model.write().overwrite().save(str(spark_model_path))
print(f'Saved validated Spark linear regression model folder to {spark_model_path}')

# 7. Clean up memory and remove temporary staging file
spark.stop()
if os.path.exists('spark_staging_buffer.csv'):
    os.remove('spark_staging_buffer.csv')
print("Spark session closed and staging files cleared safely.")


## 8) Cleanup

In [ ]:
spark.stop()
print('\n--- Pipeline Completed Successfully ---')